# Scatter GUI - Quantum Toolkit
This notebook provides an interactive GUI for quantum scattering and eigenstate calculations using ipywidgets and quantum_toolkit.

## Imports and Style
Import all required libraries and set up notebook-wide CSS for widget display.

In [1]:
import atomic_units as au
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
from quantum_toolkit import potentials as pots
import quantum_toolkit as quat
import scipy.sparse as sparse
import numpy as np
import time
import threading
import os
import json
import glob
import h5py  # For HDF5 file handling

display(HTML("""
<style>
.output_scroll {overflow: visible !important; max-height: none !important;}
.potparam-row > .widget-label { min-width: 90px; }
.potparam-row > .widget-label, .potparam-row > .widget-inline-hbox { margin-right: 8px; }
</style>
"""))

## Global Parameters and Unit System
Define global constants, atomic unit system, and grid.

In [2]:
# Global parameters
hb = 1.0
m0 = 8.8
k0 = 2.3
e0 = 3.4
unit_sys = au.AtomicUnitSystem(xh=hb, xe=e0, xm=m0, xk=k0)

# Unit system widgets
hbar_widget = widgets.FloatText(description="hbar:", value=unit_sys.hb)
m0_widget = widgets.FloatText(description="m0:", value=unit_sys.me)
kappa0_widget = widgets.FloatText(description="kappa0:", value=unit_sys.k0)
e0_widget = widgets.FloatText(description="e0:", value=unit_sys.e0)

# Grid setup
uxgrid_num = 4*1024
uxgrid_dx = 0.025
uxgrid_width = uxgrid_num * uxgrid_dx
uxgrid = np.arange(-uxgrid_width/2, uxgrid_width/2, uxgrid_dx)
dx_widget = widgets.FloatText(description="Step size:", value=uxgrid_dx, min=1e-10)

## Potential Parameters and Model Potential
Define widgets for potential parameters and the function to generate the model potential.

In [3]:
# Potential parameter widgets
wallwidth_widget = widgets.FloatText(value=3.0, description="wallwidth [nm]")
wallheight_widget = widgets.FloatText(value=1.0, description="wallheight [eV]")
wallrise_widget = widgets.FloatText(value=1.0, description="wallrise [nm]")
wellwidth_widget = widgets.FloatText(value=10.0, description="wellwidth [nm]")
welldepth_widget = widgets.FloatText(value=1.0, description="welldepth [eV]")
wellfall_widget = widgets.FloatText(value=1.0, description="wellfall [nm]")

# Superlattice widgets
num_cells_widget = widgets.IntText(value=1, min=1, description="Number of cells:", style={'description_width': 'initial'})
cell_spacing_widget = widgets.FloatText(value=0.0, description="Cell spacing [nm]:", style={'description_width': 'initial'})

def get_modelpot():
    wallwidth = wallwidth_widget.value * unit_sys.convert_length_from('nm')
    wallheight = wallheight_widget.value * unit_sys.convert_energy_from('eV')
    wallrise = wallrise_widget.value
    wellwidth = wellwidth_widget.value * unit_sys.convert_length_from('nm')
    welldepth = welldepth_widget.value * unit_sys.convert_energy_from('eV')
    wellfall = wellfall_widget.value
    num_cells = num_cells_widget.value
    cell_spacing = cell_spacing_widget.value * unit_sys.convert_length_from('nm')

    # One cell width (2*wall + well + spacing)
    cell_length_nm = 2*wallwidth_widget.value + wellwidth_widget.value + cell_spacing_widget.value
    cell_length_unit = cell_length_nm * unit_sys.convert_length_from('nm')

    pot = np.zeros_like(uxgrid)
    for i in range(num_cells):
        offset = (i - (num_cells-1)/2) * cell_length_unit
        cell_pot = pots.SmoothStackPotential(
            uxgrid - offset,
            wallwidth=wallwidth,
            wallheight=wallheight,
            welldepth=welldepth,
            wellwidth=wellwidth,
            wallrise=wallrise,
            wellfall=wellfall
        )(0)
        pot += cell_pot
    return lambda t=0: pot

## Potential Tab and Plotting
Set up the potential tab and plotting logic.

In [4]:
plot_unit_toggle = widgets.ToggleButtons(
    options=[('unit_sys', False), ('nm/eV', True)],
    value=True,
    description='Plot units:',
    style={'description_width': 'initial'}
)
plot_margin_widget = widgets.FloatText(value=10.0, layout=widgets.Layout(width='70px'))
plot_margin_label = widgets.Label("nm")
pot_output = widgets.Output()
pot_output.layout = widgets.Layout(overflow='visible', max_height='none')

def plot_potential(change=None):
    with pot_output:
        clear_output(wait=True)
        modelpot = get_modelpot()
        plot_margin_nm = plot_margin_widget.value
        total_width_nm = num_cells_widget.value * (2*wallwidth_widget.value + wellwidth_widget.value + cell_spacing_widget.value)
        x_min_nm = -0.5 * total_width_nm - plot_margin_nm
        x_max_nm = 0.5 * total_width_nm + plot_margin_nm

        if plot_unit_toggle.value:
            x_plot = uxgrid * unit_sys.length_unit.to('nm').magnitude
            y = modelpot(0) * unit_sys.energy_unit.to('eV').magnitude
            x_min = x_min_nm
            x_max = x_max_nm
            xlabel = "x [nm]"
            ylabel = "Potential [eV]"
        else:
            x_plot = uxgrid
            x_min = (x_min_nm / unit_sys.length_unit.to('nm').magnitude)
            x_max = (x_max_nm / unit_sys.length_unit.to('nm').magnitude)
            xlabel = "x [unit_sys]"
            ylabel = "Potential [unit_sys]"
            y = modelpot(0)

        plt.figure(figsize=(7, 3))
        plt.plot(x_plot, y)
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.grid()
        plt.xlim([x_min, x_max])
        plt.tight_layout()
        plt.show()

for w in [
    wallwidth_widget, wallheight_widget, wallrise_widget,
    wellwidth_widget, welldepth_widget, wellfall_widget,
    plot_unit_toggle, plot_margin_widget,
    num_cells_widget, cell_spacing_widget
]:
    w.observe(plot_potential, names='value')
plot_potential()

potential_tab = widgets.VBox([
    widgets.Label("SmoothStackPotential parameters"),
    plot_unit_toggle,
    widgets.HBox([widgets.Label("Plot margin:"), plot_margin_widget, plot_margin_label]),
    num_cells_widget,
    cell_spacing_widget,
    wallwidth_widget, wallheight_widget, wallrise_widget,
    wellwidth_widget, welldepth_widget, wellfall_widget,
    pot_output
])

## Eigenstate Calculation Tab
Widgets and logic for eigenstate calculation and plotting.

In [5]:
eigen_output = widgets.Output()

num_eigen_widget = widgets.IntText(
    value=6, min=1, description="Number of eigenvalues to compute:",
    style={'description_width': 'initial'}
)
num_plot_widget = widgets.IntText(
    value=6, min=1, description="Show on plot:",
    style={'description_width': 'initial'}
)

eigen_unit_toggle = widgets.ToggleButtons(
    options=[('unit_sys', False), ('nm/eV', True)],
    value=True,
    description='Plot units:',
    style={'description_width': 'initial'}
)

# Store last results for plotting
_last_eigenvalues = None
_last_eigenvectors = None
_last_x_plot = None
_last_y_pot = None
_last_eigenvalues_plot = None
_last_xlabel = None
_last_ylabel = None

def compute_eigenstates(_=None):
    global _last_eigenvalues, _last_eigenvectors, _last_x_plot, _last_y_pot, _last_eigenvalues_plot, _last_xlabel, _last_ylabel
    with eigen_output:
        clear_output(wait=True)
        print("Eigenstate calculation started...")
        start_time = time.time()
        wallwidth = wallwidth_widget.value * unit_sys.convert_length_from('nm')
        wallheight = wallheight_widget.value * unit_sys.convert_energy_from('eV')
        welldepth = welldepth_widget.value * unit_sys.convert_energy_from('eV')
        wellwidth = wellwidth_widget.value * unit_sys.convert_length_from('nm')
        wallrise = wallrise_widget.value
        wellfall = wellfall_widget.value
        modelpot = get_modelpot()
        zeropot = pots.ZeroPotential(uxgrid)
        ham_static = quat.HamiltonOperator(
            uxgrid,
            hbar=unit_sys.hb,
            me=unit_sys.me,
            charge=unit_sys.e0,
            scalarpot=modelpot,
            vectorpot=zeropot
        )
        num_eigen = max(1, int(num_eigen_widget.value))
        num_plot = max(1, int(num_plot_widget.value))
        print(f"Diagonalization in progress for {num_eigen} eigenvalues, please wait...")
        eigenvalues, eigenvectors = sparse.linalg.eigsh(ham_static(0), num_eigen, which='SR')
        elapsed = time.time() - start_time

        # Unit selection
        if eigen_unit_toggle.value:
            x_plot = uxgrid * unit_sys.length_unit.to('nm').magnitude
            y_pot = modelpot(0) * unit_sys.energy_unit.to('eV').magnitude
            eigenvalues_plot = eigenvalues * unit_sys.energy_unit.to('eV').magnitude
            xlabel = "x [nm]"
            ylabel = "Energy [eV]"
        else:
            x_plot = uxgrid
            y_pot = modelpot(0)
            eigenvalues_plot = eigenvalues
            xlabel = "x [unit_sys]"
            ylabel = "Energy [unit_sys]"

        # Store results
        _last_eigenvalues = eigenvalues
        _last_eigenvectors = eigenvectors
        _last_x_plot = x_plot
        _last_y_pot = y_pot
        _last_eigenvalues_plot = eigenvalues_plot
        _last_xlabel = xlabel
        _last_ylabel = ylabel

        print(f"Calculation finished in {elapsed:.1f} seconds.")
        print(f"Eigenvalues:", eigenvalues_plot[:num_plot])
        plot_eigenstates(num_plot)

def plot_eigenstates(num_plot=None):
    with eigen_output:
        clear_output(wait=True)
        if _last_eigenvalues is None:
            print("No eigenstate data. Please compute eigenstates first.")
            return
        if num_plot is None:
            num_plot = max(1, int(num_plot_widget.value))
        x_plot = _last_x_plot
        y_pot = _last_y_pot
        eigenvalues_plot = _last_eigenvalues_plot
        eigenvectors = _last_eigenvectors
        xlabel = _last_xlabel
        ylabel = _last_ylabel
        num_eigen = len(_last_eigenvalues)

        print(f"Eigenvalues:", eigenvalues_plot[:num_plot])

        plt.figure(figsize=(8, 4))
        plt.plot(x_plot, y_pot, label="Potential")
        for eval, evec in zip(eigenvalues_plot[:num_plot], np.transpose(eigenvectors)[:num_plot]):
            plt.plot(x_plot, eval * np.ones_like(x_plot), "--", label=f"E={eval:.2e}")
        plt.xlabel(xlabel)
        plt.xlim([x_plot[0], x_plot[-1]])
        plt.legend(loc='upper right')
        plt.grid()
        plt.title("Eigenstates and Potential")
        plt.show()

        plt.figure()
        plt.scatter(np.arange(num_eigen)[:num_plot], eigenvalues_plot[:num_plot], marker='o', color='red')
        plt.xlabel("State index")
        plt.ylabel(ylabel)
        plt.title("Eigenvalues")
        plt.grid()
        plt.show()

eigen_button = widgets.Button(description="Compute eigenstates", button_style='primary')
eigen_button.on_click(compute_eigenstates)
num_plot_widget.observe(lambda change: plot_eigenstates(), names='value')
eigen_unit_toggle.observe(lambda change: plot_eigenstates(), names='value')

# Photon wavelength and energy fields
photon_wavelength_widget = widgets.FloatText(
    value=800.0, description="Photon wavelength [nm]:", style={'description_width': 'initial'}
)
photon_energy_widget = widgets.Label(value="Photon energy: --- eV")

def update_photon_energy(change=None):
    # E = hc/λ, h = 4.135667696e-15 eV·s, c = 299792458 m/s, λ in nm
    h = 4.135667696e-15  # eV·s
    c = 299792458  # m/s
    try:
        wavelength_nm = photon_wavelength_widget.value
        wavelength_m = wavelength_nm * 1e-9
        energy_eV = h * c / wavelength_m
        photon_energy_widget.value = f"Photon energy: {energy_eV:.3f} eV"
    except Exception:
        photon_energy_widget.value = "Photon energy: --- eV"

photon_wavelength_widget.observe(update_photon_energy, names='value')
update_photon_energy()

eigen_tab = widgets.VBox([
    widgets.Label("Eigenstate calculation"),
    eigen_unit_toggle,
    num_eigen_widget,
    num_plot_widget,
    photon_wavelength_widget,
    photon_energy_widget,
    widgets.HBox([eigen_button]),
    eigen_output
])

## Dispersion Tab
Widgets and logic for dispersion object creation and visualization.

In [6]:
# Dispersion object and update logic
dispersion = None
def update_dispersion(*args):
    global dispersion
    try:
        dispersion = quat.CosineDispersion(
            dx_widget.value,
            hbar=unit_sys.hb,
            mass=unit_sys.me
        )
    except Exception as e:
        print("Error creating dispersion:", e)
update_dispersion()

dx_widget.observe(lambda change: update_dispersion(), names='value')
hbar_widget.observe(lambda change: update_dispersion(), names='value')
m0_widget.observe(lambda change: update_dispersion(), names='value')

dispersion_output = widgets.Output()

def create_dispersion(_=None):
    with dispersion_output:
        clear_output(wait=True)
        update_dispersion()
        print("Dispersion object created/updated:")
        print("dx =", dx_widget.value)
        print("hbar =", unit_sys.hb)
        print("mass =", unit_sys.me)

dispersion_button = widgets.Button(description="Create dispersion", button_style='primary')
dispersion_button.on_click(create_dispersion)
dispersion_tab = widgets.VBox([
    widgets.Label("Create/update dispersion object with current settings"),
    dispersion_button,
    dispersion_output
])

## Unit System and Grid Tabs
Widgets for unit system and grid setup.

In [7]:
unit_tab = widgets.VBox([
    widgets.Label("Atomic Unit system"),
    hbar_widget,
    m0_widget,
    kappa0_widget,
    e0_widget
])
points_widget = widgets.IntText(description="Number of points:", value=uxgrid_num, min=1)
dx_widget = widgets.FloatText(description="Step size:", value=uxgrid_dx, min=1e-10)
width_widget = widgets.Label(value="Total width: ---")
def update_width(*args):
    try:
        n = points_widget.value
        dx = dx_widget.value
        width = n * dx * unit_sys.length_unit.to('nm')
        width_widget.value = f"Total width: {width:.4f}"
    except Exception:
        width_widget.value = "Total width: ---"
points_widget.observe(update_width, names='value')
dx_widget.observe(update_width, names='value')
update_width()
grid_tab = widgets.VBox([
    widgets.Label("Grid setup"),
    points_widget,
    dx_widget,
    width_widget
])

## Scatter Calculation Tab
Widgets and logic for scattering calculation and plotting.

In [8]:
scatter_output = widgets.Output()
quasiparticle_selector = widgets.Dropdown(
    options=[],
    description="Quasiparticle:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)
show_wave_output = widgets.Output()
progress_bar = widgets.FloatProgress(value=0.0, min=0.0, max=1.0, description='Progress:', bar_style='info', layout=widgets.Layout(width='350px'))

# State for interrupting calculation
scatter_calc_state = {'running': False, 'thread': None}

def compute_scatter_thread():
    with scatter_output:
        clear_output(wait=True)
        t_vals = []
        r_vals = []
        quasiparticles_pk = []
        quasiparticles_mk = []
        qp_labels = []

        # Parameters from dispersion tab
        energy_min = disp_energy_min.value
        energy_max = disp_energy_max.value
        num_points = disp_number_of_points.value

        energy_min_in_unit_sys = energy_min * unit_sys.convert_energy_from('eV')
        energy_max_in_unit_sys = energy_max * unit_sys.convert_energy_from('eV')

        # Grid in k or energy
        if disp_grid_mode.value == 'unified in k':
            k_min = dispersion.wavenumber(energy_min_in_unit_sys)
            k_max = dispersion.wavenumber(energy_max_in_unit_sys)
            k_vals = np.linspace(k_min, k_max, num_points)
            e_vals = [dispersion.energy(k) for k in k_vals]
        else:
            e_vals = np.linspace(energy_min_in_unit_sys, energy_max_in_unit_sys, num_points)
            k_vals = [dispersion.wavenumber(e) for e in e_vals]

        # --- Konverzió nm/eV egységbe, ha szükséges ---
        use_nm_ev = plot_unit_toggle.value if 'plot_unit_toggle' in globals() else True
        if use_nm_ev:
            k_plot = np.array(k_vals) * unit_sys.length_unit.to('nm').magnitude**-1  # 1/nm
            e_plot = np.array(e_vals) * unit_sys.energy_unit.to('eV').magnitude      # eV
            k_xlabel = "k [1/nm]"
            e_xlabel = "electron energy [eV]"
        else:
            k_plot = np.array(k_vals)
            e_plot = np.array(e_vals)
            k_xlabel = "k [unit_sys]"
            e_xlabel = "electron energy [unit_sys]"

        # Potential and Hamiltonian
        modelpot = get_modelpot()
        zeropot = pots.ZeroPotential(uxgrid)
        ham_static = quat.HamiltonOperator(
            uxgrid,
            hbar=unit_sys.hb,
            me=unit_sys.me,
            charge=unit_sys.e0,
            scalarpot=modelpot,
            vectorpot=zeropot
        )

        for idx, (ein, kin) in enumerate(zip(e_vals, k_vals)):
            if not scatter_calc_state['running']:
                progress_bar.value = 0.0
                progress_bar.bar_style = 'warning'
                return
            qp_pk, qp_mk = quat.quasiparticles_at_given_energy(ein, kin, ham_static)
            quasiparticles_pk.append(qp_pk)
            quasiparticles_mk.append(qp_mk)
            t_vals.append(qp_pk['transmission'])
            r_vals.append(qp_pk['reflection'])
            # --- Címkék is eV/1/nm egységben, ha kell ---
            if use_nm_ev:
                label_e = ein * unit_sys.energy_unit.to('eV').magnitude
                label_k = kin * unit_sys.length_unit.to('nm').magnitude**-1
            else:
                label_e = ein
                label_k = kin
            qp_labels.append(f"E={label_e:.3e} k={label_k:.3e} T={qp_pk['transmission']:.2f}")
            progress_bar.value = (idx + 1) / num_points

            # Update dropdown and data at each step
            quasiparticle_selector.options = [(label, i) for i, label in enumerate(qp_labels)]
            quasiparticle_selector._quasiparticles = list(quasiparticles_pk)
            quasiparticle_selector._modelpot = modelpot
            if quasiparticle_selector.value is None and qp_labels:
                quasiparticle_selector.value = 0

        # --- ÚJ: Mindig frissítjük a legutolsó quasiparticle listát és címkéket ---
        quasiparticle_selector.options = [(label, i) for i, label in enumerate(qp_labels)]
        quasiparticle_selector._quasiparticles = list(quasiparticles_pk)
        quasiparticle_selector._modelpot = modelpot
        # --- VÉGE ÚJ ---

        # Plot results
        plt.figure()
        plt.plot(k_plot, t_vals, "-o", label="T")
        plt.plot(k_plot, r_vals, "-o", label="R")
        plt.xlabel(k_xlabel)
        plt.legend()
        plt.grid()
        plt.title("Transmission/Reflection vs k")
        plt.show()

        plt.figure()
        plt.plot(e_plot, t_vals, "-o", label="T")
        plt.plot(e_plot, r_vals, "-o", label="R")
        plt.xlabel(e_xlabel)
        plt.legend()
        plt.grid()
        plt.title("Transmission/Reflection vs energy")
        plt.show()

        progress_bar.value = 1.0
        progress_bar.bar_style = 'success'

def compute_scatter(_=None):
    if scatter_calc_state['running']:
        # Stop button pressed
        scatter_calc_state['running'] = False
        scatter_button.description = "Compute"
        scatter_button.button_style = 'primary'
        progress_bar.bar_style = 'danger'
        return
    # Start calculation
    scatter_calc_state['running'] = True
    scatter_button.description = "Stop"
    scatter_button.button_style = 'danger'
    progress_bar.value = 0.0
    progress_bar.bar_style = 'info'
    thread = threading.Thread(target=compute_scatter_thread)
    scatter_calc_state['thread'] = thread
    thread.start()

def plot_selected_quasiparticle(change=None):
    idx = quasiparticle_selector.value
    quasiparticles = getattr(quasiparticle_selector, "_quasiparticles", None)
    modelpot = getattr(quasiparticle_selector, "_modelpot", None)
    if quasiparticles is None or modelpot is None or idx is None:
        return
    qp = quasiparticles[idx]
    with show_wave_output:
        clear_output(wait=True)
        # --- X tengely konverzió, ha szükséges ---
        use_nm_ev = plot_unit_toggle.value if 'plot_unit_toggle' in globals() else True
        if use_nm_ev:
            x_plot = uxgrid * unit_sys.length_unit.to('nm').magnitude
            xlabel = "x [nm]"
        else:
            x_plot = uxgrid
            xlabel = "x [unit_sys]"
        plt.figure(figsize=(8, 4))
        plt.plot(x_plot, modelpot(0), label="Potential")
        plt.plot(x_plot, np.real(qp["state"]["value"]), label="Re")
        plt.plot(x_plot, np.imag(qp["state"]["value"]), label="Im")
        plt.plot(x_plot, np.abs(qp["state"]["value"]), label="Abs")
        plt.xlabel(xlabel)
        plt.legend()
        plt.grid()
        plt.title("Selected quasiparticle wavefunction")
        plt.show()

quasiparticle_selector.observe(plot_selected_quasiparticle, names='value')

scatter_button = widgets.Button(description="Compute", button_style='primary')
scatter_button.on_click(compute_scatter)
scatter_tab = widgets.VBox([
    widgets.Label("Scatter"),
    scatter_button,
    progress_bar,
    quasiparticle_selector,
    scatter_output,
    show_wave_output
])

## Dispersion Grid Visualization Tab
Interactive plot for dispersion grid setup.

In [9]:
disp_grid_mode = widgets.Dropdown(
    options=['unified in energy', 'unified in k'],
    value='unified in k',
    description='Grid type:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)
disp_energy_min = widgets.FloatText(
    value=0.02,
    description='min:',
    min=0.0,
    style={'description_width': '60px'},
    layout=widgets.Layout(width='180px')
)
disp_min_unit_label = widgets.Label(value='eV')
disp_energy_max = widgets.FloatText(
    value=1.2,
    description='max:',
    min=0.0,
    style={'description_width': '60px'},
    layout=widgets.Layout(width='180px')
)
disp_max_unit_label = widgets.Label(value='eV')
disp_number_of_points = widgets.IntText(
    value=50,
    min=2,
    description='Num. of pts.:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='180px')
)

# --- Dispersion Grid Visualization Tab ---
disp_plot_unit_toggle = widgets.ToggleButtons(
    options=[('unit_sys', False), ('nm/eV', True)],
    value=True,
    description='Plot units:',
    style={'description_width': 'initial'}
)

def update_dispersion_tab_plot(grid_mode=None, energy_min=None, energy_max=None, num=None, plot_unit_toggle=None):
    grid_mode = disp_grid_mode.value if grid_mode is None else grid_mode
    energy_min = disp_energy_min.value if energy_min is None else energy_min
    energy_max = disp_energy_max.value if energy_max is None else energy_max
    num = disp_number_of_points.value if num is None else num
    use_nm_ev = disp_plot_unit_toggle.value if plot_unit_toggle is None else plot_unit_toggle
    import matplotlib.pyplot as plt
    e_min = energy_min
    e_max = energy_max
    if dispersion is None:
        print("Dispersion object is not initialized. Please create it first with the 'Create dispersion' button!")
        return
    e_min_unit = e_min * unit_sys.convert_energy_from('eV')
    e_max_unit = e_max * unit_sys.convert_energy_from('eV')
    if grid_mode == 'unified in k':
        k_min = dispersion.wavenumber(e_min_unit)
        k_max = dispersion.wavenumber(e_max_unit)
        kvals_unit, dk = np.linspace(k_min, k_max, num, retstep=True)
        evals_unit = dispersion.energy(kvals_unit)
        if use_nm_ev:
            kvals = kvals_unit * unit_sys.length_unit.to('nm').magnitude**-1  # 1/nm
            evals = evals_unit * unit_sys.energy_unit.to('eV').magnitude      # eV
            xlabel = "k [1/nm]"
            ylabel = "energy [eV]"
        else:
            kvals = kvals_unit
            evals = evals_unit
            xlabel = "k [unit_sys]"
            ylabel = "energy [unit_sys]"
        plt.plot(kvals, evals, 'o-', label='k-grid')
        plt.scatter(kvals, np.full_like(kvals, plt.ylim()[0]), color='red', marker='|', s=100, label='k points')
        plt.scatter(np.full_like(evals, plt.xlim()[0]), evals, color='blue', marker='_', s=100, label='E points')
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.title("Equidistant in k")
    else:
        evals_unit = np.linspace(e_min_unit, e_max_unit, num)
        kvals_unit = dispersion.wavenumber(evals_unit)
        if use_nm_ev:
            kvals = kvals_unit * unit_sys.length_unit.to('nm').magnitude**-1  # 1/nm
            evals = evals_unit * unit_sys.energy_unit.to('eV').magnitude      # eV
            xlabel = "k [1/nm]"
            ylabel = "energy [eV]"
        else:
            kvals = kvals_unit
            evals = evals_unit
            xlabel = "k [unit_sys]"
            ylabel = "energy [unit_sys]"
        plt.plot(kvals, evals, 'o-', label='energy-grid')
        plt.scatter(kvals, np.full_like(kvals, plt.ylim()[0]), color='red', marker='|', s=100, label='k points')
        plt.scatter(np.full_like(evals, plt.xlim()[0]), evals, color='blue', marker='_', s=100, label='E points')
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.title("Equidistant in energy")
    plt.grid()
    plt.legend()
    plt.show()

disp_wg_plot = widgets.interactive_output(
    lambda grid_mode, energy_min, energy_max, num, plot_unit_toggle: update_dispersion_tab_plot(
        grid_mode, energy_min, energy_max, num, plot_unit_toggle
    ),
    {
        'grid_mode': disp_grid_mode,
        'energy_min': disp_energy_min,
        'energy_max': disp_energy_max,
        'num': disp_number_of_points,
        'plot_unit_toggle': disp_plot_unit_toggle
    }
)

dispersion_tab = widgets.VBox([
    widgets.Label("Dispersion grid setup"),
    disp_plot_unit_toggle,
    disp_grid_mode,
    widgets.HBox([disp_energy_min, disp_min_unit_label]),
    widgets.HBox([disp_energy_max, disp_max_unit_label]),
    disp_number_of_points,
    disp_wg_plot
])

## Scatter Plot Tab
Interactive plot for transmission/reflection after calculation.

In [10]:
scatter_plot_mode = widgets.ToggleButtons(
    options=[('k', 'k'), ('energy', 'energy')],
    value='k',
    description='X axis:',
    style={'description_width': 'initial'}
)
scatter_plot_output = widgets.Output()

def plot_scatter_transmission_reflection(change=None):
    with scatter_plot_output:
        clear_output(wait=True)
        quasiparticles = getattr(quasiparticle_selector, "_quasiparticles", None)
        if not quasiparticles or len(quasiparticles) == 0:
            print("Run the calculation in the Compute tab first!")
            return

        qp_labels = getattr(quasiparticle_selector, "options", [])
        k_vals = []
        e_vals = []
        t_vals = []
        r_vals = []
        for i, (label, idx) in enumerate(qp_labels):
            try:
                parts = label.split()
                e_val = float(parts[0].split('=')[1])
                k_val = float(parts[1].split('=')[1])
                t_val = float(parts[2].split('=')[1])
                r_val = quasiparticles[idx].get('reflection', 0)
                e_vals.append(e_val)
                k_vals.append(k_val)
                t_vals.append(t_val)
                r_vals.append(r_val)
            except Exception:
                continue

        # --- X tengely egység választás ---
        if plot_unit_toggle.value if 'plot_unit_toggle' in globals() else True:
            k_xlabel = "k [1/nm]"
            e_xlabel = "electron energy [eV]"
        else:
            k_xlabel = "k [unit_sys]"
            e_xlabel = "electron energy [unit_sys]"

        if scatter_plot_mode.value == 'k':
            plt.plot(k_vals, t_vals, "-o", label="T")
            plt.plot(k_vals, r_vals, "-o", label="R")
            plt.xlabel(k_xlabel)
        else:
            plt.plot(e_vals, t_vals, "-o", label="T")
            plt.plot(e_vals, r_vals, "-o", label="R")
            plt.xlabel(e_xlabel)
        plt.legend()
        plt.grid()
        plt.ylabel("Coefficient")
        plt.title("Transmission/Reflection")
        plt.show()

scatter_plot_mode.observe(plot_scatter_transmission_reflection, names='value')
plot_scatter_transmission_reflection()

scatter_tab2 = widgets.VBox([
    widgets.Label("Scatter plot"),
    scatter_plot_mode,
    scatter_plot_output
])

## Fájlok fül
Adja meg a mappa elérési útját és a fájl nevét. A beállítások automatikusan mentésre kerülnek a temporary settings file-ba.

In [11]:
import os
import ipywidgets as widgets
from IPython.display import clear_output

files_output = widgets.Output()

# Új: mappa és fájlnév mezők
directory_widget = widgets.Text(
    value=os.getcwd(),
    description="Mappa:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)
filename_widget = widgets.Text(
    value="scatter_gui_data.h5",
    description="Fájlnév:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

def get_hdf5_path():
    return os.path.join(directory_widget.value, filename_widget.value)

save_hdf5_button = widgets.Button(description="Mentés HDF5-be", button_style='success')
load_hdf5_button = widgets.Button(description="Betöltés HDF5-ből", button_style='warning')

SETTINGS_FILE = "/tmp/scatter_gui_settings.json"

def save_to_hdf5_clicked(b):
    with files_output:
        clear_output(wait=True)
        try:
            import numpy as np
            path = get_hdf5_path()
            os.makedirs(os.path.dirname(path), exist_ok=True)
            with h5py.File(path, "w") as f:
                f.attrs["created_by"] = "scatter_gui"
                # Paraméterek mentése
                params = {
                    "wallwidth": wallwidth_widget.value,
                    "wallheight": wallheight_widget.value,
                    "wallrise": wallrise_widget.value,
                    "wellwidth": wellwidth_widget.value,
                    "welldepth": welldepth_widget.value,
                    "wellfall": wellfall_widget.value,
                    "num_cells": num_cells_widget.value,
                    "cell_spacing": cell_spacing_widget.value,
                    "plot_margin": plot_margin_widget.value,
                    "plot_unit_toggle": plot_unit_toggle.value,
                    "num_eigen": num_eigen_widget.value,
                    "num_plot": num_plot_widget.value,
                    "eigen_unit_toggle": eigen_unit_toggle.value,
                    "photon_wavelength": photon_wavelength_widget.value,
                    "hbar": hbar_widget.value,
                    "m0": m0_widget.value,
                    "kappa0": kappa0_widget.value,
                    "e0": e0_widget.value,
                    "uxgrid_num": points_widget.value,
                    "uxgrid_dx": dx_widget.value,
                    "disp_grid_mode": disp_grid_mode.value,
                    "disp_energy_min": disp_energy_min.value,
                    "disp_energy_max": disp_energy_max.value,
                    "disp_number_of_points": disp_number_of_points.value,
                    "scatter_plot_mode": scatter_plot_mode.value,
                    "directory": directory_widget.value,
                    "filename": filename_widget.value
                }
                for k, v in params.items():
                    f.attrs[k] = v

                # Potenciál és rács mentése
                f.create_dataset("uxgrid", data=uxgrid)
                pot = get_modelpot()(0)
                f.create_dataset("potential", data=pot)

                # Eigenstate adatok mentése, ha vannak
                if _last_eigenvalues is not None and _last_eigenvectors is not None:
                    f.create_dataset("eigenvalues", data=_last_eigenvalues)
                    f.create_dataset("eigenvectors", data=_last_eigenvectors)
                    f.create_dataset("eigen_x_plot", data=_last_x_plot if _last_x_plot is not None else np.zeros_like(uxgrid))
                    f.create_dataset("eigen_y_pot", data=_last_y_pot if _last_y_pot is not None else np.zeros_like(uxgrid))
                    f.create_dataset("eigenvalues_plot", data=_last_eigenvalues_plot if _last_eigenvalues_plot is not None else np.zeros_like(_last_eigenvalues))
                    # xlabel/ylabel stringeket is mentjük
                    f.attrs["eigen_xlabel"] = _last_xlabel if _last_xlabel is not None else ""
                    f.attrs["eigen_ylabel"] = _last_ylabel if _last_ylabel is not None else ""

                # Scatter adatok mentése, ha vannak
                quasiparticles = getattr(quasiparticle_selector, "_quasiparticles", None)
                qp_labels = getattr(quasiparticle_selector, "options", [])
                if quasiparticles and len(quasiparticles) > 0:
                    grp = f.create_group("scatter")
                    k_vals = []
                    e_vals = []
                    t_vals = []
                    r_vals = []
                    for i, (label, idx) in enumerate(qp_labels):
                        try:
                            parts = label.split()
                            e_val = float(parts[0].split('=')[1])
                            k_val = float(parts[1].split('=')[1])
                            t_val = float(parts[2].split('=')[1])
                            r_val = quasiparticles[idx].get('reflection', 0)
                            e_vals.append(e_val)
                            k_vals.append(k_val)
                            t_vals.append(t_val)
                            r_vals.append(r_val)
                        except Exception:
                            continue
                    grp.create_dataset("k_vals", data=np.array(k_vals))
                    grp.create_dataset("e_vals", data=np.array(e_vals))
                    grp.create_dataset("t_vals", data=np.array(t_vals))
                    grp.create_dataset("r_vals", data=np.array(r_vals))
                    # Minden quasiparticle összes adatát elmentjük külön csoportba
                    qp_group = grp.create_group("quasiparticles")
                    for i, qp in enumerate(quasiparticles):
                        qpg = qp_group.create_group(str(i))
                        for key, val in qp.items():
                            # Ha a value egy dict (pl. state), azt is rekurzívan mentjük
                            if isinstance(val, dict):
                                subg = qpg.create_group(key)
                                for sk, sv in val.items():
                                    try:
                                        subg.create_dataset(sk, data=np.array(sv))
                                    except Exception:
                                        # Ha nem menthető, stringként mentjük
                                        subg.attrs[sk] = str(sv)
                            else:
                                try:
                                    qpg.create_dataset(key, data=np.array(val))
                                except Exception:
                                    qpg.attrs[key] = str(val)
                print(f"Sikeres mentés: {path}")
        except Exception as e:
            print(f"Hiba mentés közben: {e}")

def load_from_hdf5_clicked(b):
    with files_output:
        clear_output(wait=True)
        try:
            import numpy as np
            path = get_hdf5_path()
            if not os.path.exists(path):
                print(f"Nem található a fájl: {path}")
                return
            with h5py.File(path, "r") as f:
                # Paraméterek visszatöltése
                for k in [
                    "wallwidth", "wallheight", "wallrise", "wellwidth", "welldepth", "wellfall",
                    "num_cells", "cell_spacing", "plot_margin", "plot_unit_toggle", "num_eigen", "num_plot",
                    "eigen_unit_toggle", "photon_wavelength", "hbar", "m0", "kappa0", "e0",
                    "uxgrid_num", "uxgrid_dx", "disp_grid_mode", "disp_energy_min", "disp_energy_max",
                    "disp_number_of_points", "scatter_plot_mode", "directory", "filename"
                ]:
                    if k in f.attrs:
                        v = f.attrs[k]
                        # Widget típus alapján konvertálunk
                        try:
                            if k in ["plot_unit_toggle", "eigen_unit_toggle"]:
                                v = bool(v)
                            elif k in ["num_cells", "num_eigen", "num_plot", "uxgrid_num", "disp_number_of_points"]:
                                v = int(v)
                            elif k in ["wallwidth", "wallheight", "wallrise", "wellwidth", "welldepth", "wellfall",
                                       "cell_spacing", "plot_margin", "photon_wavelength", "hbar", "m0", "kappa0", "e0",
                                       "uxgrid_dx", "disp_energy_min", "disp_energy_max"]:
                                v = float(v)
                            elif k in ["disp_grid_mode", "scatter_plot_mode", "directory", "filename"]:
                                v = str(v)
                        except Exception:
                            pass
                        # Widget beállítás
                        try:
                            if k == "wallwidth": wallwidth_widget.value = v
                            elif k == "wallheight": wallheight_widget.value = v
                            elif k == "wallrise": wallrise_widget.value = v
                            elif k == "wellwidth": wellwidth_widget.value = v
                            elif k == "welldepth": welldepth_widget.value = v
                            elif k == "wellfall": wellfall_widget.value = v
                            elif k == "num_cells": num_cells_widget.value = v
                            elif k == "cell_spacing": cell_spacing_widget.value = v
                            elif k == "plot_margin": plot_margin_widget.value = v
                            elif k == "plot_unit_toggle": plot_unit_toggle.value = v
                            elif k == "num_eigen": num_eigen_widget.value = v
                            elif k == "num_plot": num_plot_widget.value = v
                            elif k == "eigen_unit_toggle": eigen_unit_toggle.value = v
                            elif k == "photon_wavelength": photon_wavelength_widget.value = v
                            elif k == "hbar": hbar_widget.value = v
                            elif k == "m0": m0_widget.value = v
                            elif k == "kappa0": kappa0_widget.value = v
                            elif k == "e0": e0_widget.value = v
                            elif k == "uxgrid_num": points_widget.value = v
                            elif k == "uxgrid_dx": dx_widget.value = v
                            elif k == "disp_grid_mode": disp_grid_mode.value = v
                            elif k == "disp_energy_min": disp_energy_min.value = v
                            elif k == "disp_energy_max": disp_energy_max.value = v
                            elif k == "disp_number_of_points": disp_number_of_points.value = v
                            elif k == "scatter_plot_mode": scatter_plot_mode.value = v
                            elif k == "directory": directory_widget.value = v
                            elif k == "filename": filename_widget.value = v
                        except Exception:
                            pass

                # Potenciál és rács visszatöltése (csak megjelenítéshez, a widgetekből újra generálódik)
                # uxgrid = f["uxgrid"][:]
                # pot = f["potential"][:]

                # Eigenstate adatok visszatöltése
                global _last_eigenvalues, _last_eigenvectors, _last_x_plot, _last_y_pot, _last_eigenvalues_plot, _last_xlabel, _last_ylabel
                if "eigenvalues" in f and "eigenvectors" in f:
                    _last_eigenvalues = f["eigenvalues"][:]
                    _last_eigenvectors = f["eigenvectors"][:]
                    _last_x_plot = f["eigen_x_plot"][:] if "eigen_x_plot" in f else None
                    _last_y_pot = f["eigen_y_pot"][:] if "eigen_y_pot" in f else None
                    _last_eigenvalues_plot = f["eigenvalues_plot"][:] if "eigenvalues_plot" in f else None
                    _last_xlabel = f.attrs.get("eigen_xlabel", "")
                    _last_ylabel = f.attrs.get("eigen_ylabel", "")

                # Scatter adatok visszatöltése
                if "scatter" in f:
                    grp = f["scatter"]
                    k_vals = grp["k_vals"][:]
                    e_vals = grp["e_vals"][:]
                    t_vals = grp["t_vals"][:]
                    r_vals = grp["r_vals"][:]
                    # Quasiparticle dict-ek visszatöltése a quasiparticles csoportból
                    quasiparticles = []
                    qp_labels = []
                    if "quasiparticles" in grp:
                        qp_group = grp["quasiparticles"]
                        for i in range(len(qp_group)):
                            qpg = qp_group[str(i)]
                            qp = {}
                            for key in qpg:
                                if isinstance(qpg[key], h5py.Group):
                                    # dict típusú érték (pl. state)
                                    subg = qpg[key]
                                    subdict = {}
                                    for sk in subg:
                                        subdict[sk] = subg[sk][()]
                                    # attribútumokat is visszatöltjük, ha vannak
                                    for sk in subg.attrs:
                                        subdict[sk] = subg.attrs[sk]
                                    qp[key] = subdict
                                else:
                                    qp[key] = qpg[key][()]
                            # attribútumokat is visszatöltjük, ha vannak
                            for key in qpg.attrs:
                                qp[key] = qpg.attrs[key]
                            quasiparticles.append(qp)
                            # Címke generálása
                            t_val = qp.get("transmission", 0)
                            e_val = qp.get("energy", None)
                            k_val = qp.get("k", None)
                            # Ha nincs energy/k, akkor a mentett e_vals/k_vals alapján
                            if e_val is None and i < len(e_vals):
                                e_val = e_vals[i]
                            if k_val is None and i < len(k_vals):
                                k_val = k_vals[i]
                            qp_labels.append(f"E={e_val:.3e} k={k_val:.3e} T={t_val:.2f}")
                    else:
                        # Régi formátum: csak hullámfüggvény
                        for i in range(len(k_vals)):
                            qp = {
                                "transmission": t_vals[i],
                                "reflection": r_vals[i],
                                "state": {"value": grp["wavefunction"][:] if "wavefunction" in grp else np.zeros_like(uxgrid)}
                            }
                            quasiparticles.append(qp)
                            qp_labels.append(f"E={e_vals[i]:.3e} k={k_vals[i]:.3e} T={t_vals[i]:.2f}")
                    quasiparticle_selector.options = [(label, i) for i, label in enumerate(qp_labels)]
                    quasiparticle_selector._quasiparticles = quasiparticles
                    quasiparticle_selector._modelpot = get_modelpot()
                    if qp_labels:
                        quasiparticle_selector.value = 0

            print(f"Sikeres betöltés: {get_hdf5_path()}")
            # Frissítések
            plot_potential()
            update_width()
            update_photon_energy()
            update_dispersion()
            create_dispersion()
            plot_scatter_transmission_reflection()
            plot_eigenstates()
        except Exception as e:
            print(f"Hiba betöltés közben: {e}")

In [12]:
# Fájlok fül widget összeállítása (helyesen, a notebook végére kell tenni a definíciót, mielőtt a tabs változót használjuk)
files_tab = widgets.VBox([
    widgets.Label("Fájlkezelés"),
    directory_widget,
    filename_widget,
    widgets.HBox([save_hdf5_button, load_hdf5_button]),
    files_output
])

# Gombok eseménykezelőinek összekapcsolása
save_hdf5_button.on_click(save_to_hdf5_clicked)
load_hdf5_button.on_click(load_from_hdf5_clicked)

In [13]:
tabs = widgets.Tab(children=[
    unit_tab, grid_tab, potential_tab, eigen_tab,
    dispersion_tab, scatter_tab, scatter_tab2, files_tab
])
tab_titles = [
    'Unit system', 'Grid', 'Potential', 'Eigenstates',
    'Dispersion', 'Compute', 'Scatter', 'Files'
]
for i, title in enumerate(tab_titles):
    tabs.set_title(i, title)

SETTINGS_FILE = "/tmp/scatter_gui_settings.json"

def save_settings_to_file():
    settings = {
        "wallwidth": wallwidth_widget.value,
        "wallheight": wallheight_widget.value,
        "wallrise": wallrise_widget.value,
        "wellwidth": wellwidth_widget.value,
        "welldepth": welldepth_widget.value,
        "wellfall": wellfall_widget.value,
        "num_cells": num_cells_widget.value,
        "cell_spacing": cell_spacing_widget.value,
        "plot_margin": plot_margin_widget.value,
        "plot_unit_toggle": plot_unit_toggle.value,
        "num_eigen": num_eigen_widget.value,
        "num_plot": num_plot_widget.value,
        "eigen_unit_toggle": eigen_unit_toggle.value,
        "photon_wavelength": photon_wavelength_widget.value,
        "hbar": hbar_widget.value,
        "m0": m0_widget.value,
        "kappa0": kappa0_widget.value,
        "e0": e0_widget.value,
        "uxgrid_num": points_widget.value,
        "uxgrid_dx": dx_widget.value,
        "disp_grid_mode": disp_grid_mode.value,
        "disp_energy_min": disp_energy_min.value,
        "disp_energy_max": disp_energy_max.value,
        "disp_number_of_points": disp_number_of_points.value,
        "scatter_plot_mode": scatter_plot_mode.value,
        # Új: mappa és fájlnév mentése
        "directory": directory_widget.value,
        "filename": filename_widget.value
    }
    try:
        with open(SETTINGS_FILE, "w") as f:
            json.dump(settings, f)
    except Exception as e:
        print("Nem sikerült menteni a beállításokat:", e)

def load_settings_from_file():
    if not os.path.exists(SETTINGS_FILE):
        return
    try:
        with open(SETTINGS_FILE, "r") as f:
            settings = json.load(f)
        wallwidth_widget.value = settings.get("wallwidth", wallwidth_widget.value)
        wallheight_widget.value = settings.get("wallheight", wallheight_widget.value)
        wallrise_widget.value = settings.get("wallrise", wallrise_widget.value)
        wellwidth_widget.value = settings.get("wellwidth", wellwidth_widget.value)
        welldepth_widget.value = settings.get("welldepth", welldepth_widget.value)
        wellfall_widget.value = settings.get("wellfall", wellfall_widget.value)
        num_cells_widget.value = settings.get("num_cells", num_cells_widget.value)
        cell_spacing_widget.value = settings.get("cell_spacing", cell_spacing_widget.value)
        plot_margin_widget.value = settings.get("plot_margin", plot_margin_widget.value)
        plot_unit_toggle.value = settings.get("plot_unit_toggle", plot_unit_toggle.value)
        num_eigen_widget.value = settings.get("num_eigen", num_eigen_widget.value)
        num_plot_widget.value = settings.get("num_plot", num_plot_widget.value)
        eigen_unit_toggle.value = settings.get("eigen_unit_toggle", eigen_unit_toggle.value)
        photon_wavelength_widget.value = settings.get("photon_wavelength", photon_wavelength_widget.value)
        hbar_widget.value = settings.get("hbar", hbar_widget.value)
        m0_widget.value = settings.get("m0", m0_widget.value)
        kappa0_widget.value = settings.get("kappa0", kappa0_widget.value)
        e0_widget.value = settings.get("e0", e0_widget.value)
        points_widget.value = settings.get("uxgrid_num", points_widget.value)
        dx_widget.value = settings.get("uxgrid_dx", dx_widget.value)
        disp_grid_mode.value = settings.get("disp_grid_mode", disp_grid_mode.value)
        disp_energy_min.value = settings.get("disp_energy_min", disp_energy_min.value)
        disp_energy_max.value = settings.get("disp_energy_max", disp_energy_max.value)
        disp_number_of_points.value = settings.get("disp_number_of_points", disp_number_of_points.value)
        scatter_plot_mode.value = settings.get("scatter_plot_mode", scatter_plot_mode.value)
        # Új: mappa és fájlnév betöltése
        directory_widget.value = settings.get("directory", directory_widget.value)
        filename_widget.value = settings.get("filename", filename_widget.value)

        # Frissítések
        plot_potential()
        update_width()
        update_photon_energy()
        update_dispersion()
        create_dispersion()
        plot_scatter_transmission_reflection()
        plot_eigenstates()

        # Explicit observer trigger
        for w in [
            wallwidth_widget, wallheight_widget, wallrise_widget,
            wellwidth_widget, welldepth_widget, wellfall_widget,
            num_cells_widget, cell_spacing_widget,
            plot_margin_widget, plot_unit_toggle,
            num_eigen_widget, num_plot_widget, eigen_unit_toggle,
            photon_wavelength_widget,
            hbar_widget, m0_widget, kappa0_widget, e0_widget,
            points_widget, dx_widget,
            disp_grid_mode, disp_energy_min, disp_energy_max, disp_number_of_points,
            scatter_plot_mode,
            directory_widget, filename_widget
        ]:
            if hasattr(w, 'value'):
                w.value = w.value

    except Exception as e:
        print("Nem sikerült betölteni a beállításokat:", e)

# Induláskor betöltjük a beállításokat
load_settings_from_file()

# Bármely widget változásakor mentjük a beállításokat
for w in [
    wallwidth_widget, wallheight_widget, wallrise_widget,
    wellwidth_widget, welldepth_widget, wellfall_widget,
    num_cells_widget, cell_spacing_widget,
    plot_margin_widget, plot_unit_toggle,
    num_eigen_widget, num_plot_widget, eigen_unit_toggle,
    photon_wavelength_widget,
    hbar_widget, m0_widget, kappa0_widget, e0_widget,
    points_widget, dx_widget,
    disp_grid_mode, disp_energy_min, disp_energy_max, disp_number_of_points,
    scatter_plot_mode,
    directory_widget, filename_widget
]:
    w.observe(lambda change: save_settings_to_file(), names='value')

display(tabs)